In [1]:
print("ok")

ok


In [2]:
import evidently
print(evidently.__version__)

0.7.12


In [3]:
import pandas as pd                          # Import pandas for data manipulation
from evidently import Report                 # Import the Report class (main object to build and run reports in Evidently)

from evidently.presets import DataDriftPreset  # Import a preset: DataDriftPreset automatically checks drift for all columns
                                               # It bundles multiple drift metrics (numeric: KS-test, categorical: Chi-square, etc.)

from evidently.metrics import ValueDrift       # Import ValueDrift: lets you check drift for a specific column (feature or target)
                                               # Useful if you want column-level drift checks instead of the whole dataset

In [4]:
#check what inside module
import evidently

# List everything in the 'evidently' module
print(dir(evidently))

['BinaryClassification', 'ColumnType', 'DataDefinition', 'Dataset', 'LLMClassification', 'MulticlassClassification', 'Recsys', 'Regression', 'Report', 'Run', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_jupyter_nbextension_paths', '_pydantic_compat', '_registry', '_version', 'compare', 'core', 'errors', 'legacy', 'llm', 'metrics', 'nbextension', 'presets', 'pydantic_utils', 'sdk', 'tests', 'ui', 'utils', 'version_info']


In [5]:
#Check what’s inside a submodule
from evidently import presets

# See what’s available in 'presets'
print(dir(presets))

['ClassificationDummyQuality', 'ClassificationPreset', 'ClassificationQuality', 'ClassificationQualityByLabel', 'DataDriftPreset', 'DataSummaryPreset', 'DatasetStats', 'RegressionDummyQuality', 'RegressionPreset', 'RegressionQuality', 'TextEvals', 'ValueStats', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'classification', 'dataset_stats', 'drift', 'regression']


In [6]:
import evidently

# List everything in the 'presets' submodule
print(dir(evidently.presets))

['ClassificationDummyQuality', 'ClassificationPreset', 'ClassificationQuality', 'ClassificationQualityByLabel', 'DataDriftPreset', 'DataSummaryPreset', 'DatasetStats', 'RegressionDummyQuality', 'RegressionPreset', 'RegressionQuality', 'TextEvals', 'ValueStats', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'classification', 'dataset_stats', 'drift', 'regression']


In [7]:
#load boston data
data_url="https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
boston_frame=pd.read_csv(data_url)
boston_frame.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [8]:
boston_frame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     506 non-null    float64
 1   zn       506 non-null    float64
 2   indus    506 non-null    float64
 3   chas     506 non-null    int64  
 4   nox      506 non-null    float64
 5   rm       506 non-null    float64
 6   age      506 non-null    float64
 7   dis      506 non-null    float64
 8   rad      506 non-null    int64  
 9   tax      506 non-null    int64  
 10  ptratio  506 non-null    float64
 11  b        506 non-null    float64
 12  lstat    506 non-null    float64
 13  medv     506 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 55.5 KB


data drift dashboard

In [9]:
boston_frame.shape

(506, 14)

In [15]:
ref_df=boston_frame[:200]  #train/ reference data
ref_df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [16]:
cur_df=boston_frame[200:] #test data 
cur_df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
200,0.01778,95.0,1.47,0,0.4030,7.135,13.9,7.6534,3,402,17.0,384.30,4.45,32.9
201,0.03445,82.5,2.03,0,0.4150,6.162,38.4,6.2700,2,348,14.7,393.77,7.43,24.1
202,0.02177,82.5,2.03,0,0.4150,7.610,15.7,6.2700,2,348,14.7,395.38,3.11,42.3
203,0.03510,95.0,2.68,0,0.4161,7.853,33.2,5.1180,4,224,14.7,392.78,3.81,48.5
204,0.02009,95.0,2.68,0,0.4161,8.034,31.9,5.1180,4,224,14.7,390.55,2.88,50.0


In [17]:
boston_frame.columns

Index(['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax',
       'ptratio', 'b', 'lstat', 'medv'],
      dtype='object')

In [18]:
# Build the report with overall and column-level drift
report = Report(metrics=[
    DataDriftPreset(),                 # overall dataset drift
    ValueDrift(column="crim"),         # numeric column example
    ValueDrift(column="rm"),           # numeric column example
    ValueDrift(column="chas"),         # categorical column example
    ValueDrift(column="medv")          # target column drift
])

# Run the report
result = report.run(reference_data=ref_df, current_data=cur_df)

# Save to HTML
result.save_html("boston_data_drift_report.html")

# Optional: display first few rows of reference and current
print("Reference data sample:")
print(ref_df.head())
print("\nCurrent data sample:")
print(cur_df.head())

c:\Users\shaya\anaconda3\envs\visa\Lib\site-packages\scipy\stats\_stats_py.py:7400: RuntimeWarning:

divide by zero encountered in divide

c:\Users\shaya\anaconda3\envs\visa\Lib\site-packages\scipy\stats\_stats_py.py:7400: RuntimeWarning:

divide by zero encountered in divide

c:\Users\shaya\anaconda3\envs\visa\Lib\site-packages\scipy\stats\_stats_py.py:7400: RuntimeWarning:

divide by zero encountered in divide



Reference data sample:
      crim    zn  indus  chas    nox     rm   age     dis  rad  tax  ptratio  \
0  0.00632  18.0   2.31     0  0.538  6.575  65.2  4.0900    1  296     15.3   
1  0.02731   0.0   7.07     0  0.469  6.421  78.9  4.9671    2  242     17.8   
2  0.02729   0.0   7.07     0  0.469  7.185  61.1  4.9671    2  242     17.8   
3  0.03237   0.0   2.18     0  0.458  6.998  45.8  6.0622    3  222     18.7   
4  0.06905   0.0   2.18     0  0.458  7.147  54.2  6.0622    3  222     18.7   

        b  lstat  medv  
0  396.90   4.98  24.0  
1  396.90   9.14  21.6  
2  392.83   4.03  34.7  
3  394.63   2.94  33.4  
4  396.90   5.33  36.2  

Current data sample:
        crim    zn  indus  chas     nox     rm   age     dis  rad  tax  \
200  0.01778  95.0   1.47     0  0.4030  7.135  13.9  7.6534    3  402   
201  0.03445  82.5   2.03     0  0.4150  6.162  38.4  6.2700    2  348   
202  0.02177  82.5   2.03     0  0.4150  7.610  15.7  6.2700    2  348   
203  0.03510  95.0   2.68   